In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import subprocess

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

# tmp_clusts

tmp_clusts = ['Start', 'Delayed', 'Fail1']

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/rna_cluster/fail_traj/start_delayed/_rms_results_collapsed.tsv',
                   header=0, sep='\t')
dms = 1
start_delayed_dmrs = dmrs[dmrs['number_of_dms'] > dms].copy()
start_delayed_dmrs.shape

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/rna_cluster/fail_traj/delayed_fail1/_rms_results_collapsed.tsv',
                   header=0, sep='\t')
dms = 1
delayed_fail1_dmrs = dmrs[dmrs['number_of_dms'] > dms].copy()
delayed_fail1_dmrs.shape

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/rna_cluster/fail_traj/start_fail1/_rms_results_collapsed.tsv',
                   header=0, sep='\t')
dms = 1
start_fail1_dmrs = dmrs[dmrs['number_of_dms'] > dms].copy()
start_fail1_dmrs.shape

In [ ]:
# start_delayed_dmrs.shape

In [ ]:
# delayed_fail1_dmrs.shape

In [ ]:
# delayed_fail1_dmrs.head()

In [ ]:
# start_fail1_dmrs.shape

In [ ]:
# start_fail1_dmrs.head()

In [ ]:
s_d_s_hypo = start_delayed_dmrs[start_delayed_dmrs['hypomethylated_samples'] == 'Start']
s_d_d_hypo = start_delayed_dmrs[start_delayed_dmrs['hypomethylated_samples'] == 'Delayed']

d_f_d_hypo = delayed_fail1_dmrs[delayed_fail1_dmrs['hypomethylated_samples'] == 'Delayed']
d_f_f_hypo = delayed_fail1_dmrs[delayed_fail1_dmrs['hypomethylated_samples'] == 'Fail1']

s_f_s_hypo = start_fail1_dmrs[start_fail1_dmrs['hypomethylated_samples'] == 'Start']
s_f_f_hypo = start_fail1_dmrs[start_fail1_dmrs['hypomethylated_samples'] == 'Fail1']

In [ ]:
s_d_s_hypo.shape, s_d_d_hypo.shape

In [ ]:
d_f_d_hypo.shape, d_f_f_hypo.shape

In [ ]:
s_f_s_hypo.shape, s_f_f_hypo.shape

In [ ]:
def process_bed(s):
    pmds_mc = pd.read_csv(f'{projdir}/bed/features/rna_cluster/{s}.start_delayed_dmrs_CGN-Both.full.bed.gz', sep='\t', header=None)
    pmds_mc.index = [f'{x}_{y}' for x,y in zip(pmds_mc[0], pmds_mc[1])]
    return pmds_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, tmp_clusts), total=len(tmp_clusts)))
    
mc_results = [x[3] / x[4] for x in results]
s_d_df = pd.concat(mc_results, axis=1)
s_d_df.columns = tmp_clusts

for s in tmp_clusts:
    s_d_df[s].fillna(s_d_df[s].mean(), inplace=True)
    
s_d_df = s_d_df[~s_d_df.index.duplicated()]
s_d_df.shape

In [ ]:
def process_bed(s):
    pmds_mc = pd.read_csv(f'{projdir}/bed/features/rna_cluster/{s}.delayed_fail1_dmrs_CGN-Both.full.bed.gz', sep='\t', header=None)
    pmds_mc.index = [f'{x}_{y}' for x,y in zip(pmds_mc[0], pmds_mc[1])]
    return pmds_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, tmp_clusts), total=len(tmp_clusts)))
    
mc_results = [x[3] / x[4] for x in results]
d_f_df = pd.concat(mc_results, axis=1)
d_f_df.columns = tmp_clusts

for s in tmp_clusts:
    d_f_df[s].fillna(d_f_df[s].mean(), inplace=True)
    
d_f_df = d_f_df[~d_f_df.index.duplicated()]
d_f_df.shape

In [ ]:
def process_bed(s):
    pmds_mc = pd.read_csv(f'{projdir}/bed/features/rna_cluster/{s}.start_fail1_dmrs_CGN-Both.sparse.bed.gz', sep='\t', header=None)
    pmds_mc.index = [f'{x}_{y}' for x,y in zip(pmds_mc[0], pmds_mc[1])]
    return pmds_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, tmp_clusts), total=len(tmp_clusts)))
    
mc_results = [x[3] / x[4] for x in results]
s_f_df = pd.concat(mc_results, axis=1)
s_f_df.columns = tmp_clusts

for s in tmp_clusts:
    s_f_df[s].fillna(s_f_df[s].mean(), inplace=True)
    
s_f_df = s_f_df[~s_f_df.index.duplicated()]
s_f_df.shape

In [ ]:
s_f_df.head()

In [ ]:
k = 6
features = s_d_df.select_dtypes(include='number')
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features)
s_d_df['kmeans_cluster'] = cluster_labels
s_d_df['kmeans_cluster'].value_counts().sort_index()

final_s_d = s_d_df.sort_values(by=['kmeans_cluster']).iloc[:,:-1]
final_s_d.head()

In [ ]:
fig, ax = plt.subplots(1, figsize=(8,4))
ax = sns.heatmap(final_s_d, cmap='coolwarm', vmin=0.2, vmax=0.8)
ax.set_yticklabels([]);

In [ ]:
k = 6
features = d_f_df.select_dtypes(include='number')
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features)
d_f_df['kmeans_cluster'] = cluster_labels
d_f_df['kmeans_cluster'].value_counts().sort_index()

final_d_f = d_f_df.sort_values(by=['kmeans_cluster']).iloc[:,:-1]
final_d_f.head()

In [ ]:
fig, ax = plt.subplots(1, figsize=(8,4))
ax = sns.heatmap(final_d_f, cmap='coolwarm', vmin=0.2, vmax=0.8)
ax.set_yticklabels([]);

In [ ]:
k = 6
features = s_f_df.select_dtypes(include='number')
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features)
s_f_df['kmeans_cluster'] = cluster_labels
s_f_df['kmeans_cluster'].value_counts().sort_index()

final_s_f = s_f_df.sort_values(by=['kmeans_cluster']).iloc[:,:-1]
final_s_f.head()

In [ ]:
fig, ax = plt.subplots(1, figsize=(8,4))
ax = sns.heatmap(final_s_f, cmap='coolwarm', vmin=0.2, vmax=0.8)
ax.set_yticklabels([]);

### get dist

In [ ]:
final_s_d.head()

In [ ]:
tmp_dist = final_s_d['Start'] - final_s_d['Fail1']
sns.histplot(tmp_dist)

### write

In [ ]:
fig, ax = plt.subplots(1, figsize=(8,4))
ax = sns.heatmap(final_s_d, cmap='coolwarm', vmin=0.2, vmax=0.8)
ax.set_yticklabels([]);

In [ ]:
tmp_order = [2, 0, 5, 3, 1, 4]
s_d_df['kmeans_cluster'] = pd.Categorical(
    s_d_df['kmeans_cluster'],
    categories=tmp_order,
    ordered=True
)

s_d_df_sorted = s_d_df.sort_values(by='kmeans_cluster')
s_d_df_sorted

In [ ]:
tmp_plot = s_d_df_sorted.iloc[:,:-1]
fig, ax = plt.subplots(1, figsize=(8,4))
ax = sns.heatmap(tmp_plot, cmap='coolwarm', vmin=0.2, vmax=0.8)
ax.set_yticklabels([]);

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure4'
tmp_plot.to_csv(f'{outdir}/fail1_traj_reversal_dmrs.csv', sep='\t')

In [ ]:
!date